#Speech Emotion Recognition with MLP Classifier



#Dataset
The Ryerson Audio-Visual Database of Emotional Speech and Song (RAVDESS)

---
Audio-only files

Audio-only files of all actors (01-24) are available as two separate zip files (~200 MB each):

Speech file (Audio_Speech_Actors_01-24.zip, 215 MB) contains 1440 files: 60 trials per actor x 24 actors = 1440.
Song file (Audio_Song_Actors_01-24.zip, 198 MB) contains 1012 files: 44 trials per actor x 23 actors = 1012.

Total=2452

---

---
Toronto emotional speech set (TESS)

---


There are a set of 200 target words were spoken in the carrier phrase "Say the word _' by two actresses (aged 26 and 64 years) and recordings were made of the set portraying each of seven emotions (anger, disgust, fear, happiness, pleasant surprise, sadness, and neutral). There are 2800 data points (audio files) in total.

The dataset is organised such that each of the two female actor and their emotions are contain within its own folder. And within that, all 200 target words audio file can be found. The format of the audio file is a WAV format


---



# Mount google drive



In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ejlok1/cremad")

print("Path to dataset files:", path)

100%|██████████| 451M/451M [00:17<00:00, 27.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ejlok1/cremad/versions/1


# Install following libraries

In [5]:
!pip install librosa soundfile numpy sklearn pyaudio

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [6]:
!pip install soundfile

In [7]:
!pip install resampy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 9.8 MB/s eta 0:00:00


# Make the necessary imports

In [8]:
import librosa
import soundfile
import os, glob, pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

Define a function extract_feature to extract the mfcc, chroma, and mel features from a sound file. This function takes 4 parameters- the file name and three Boolean parameters for the three features:

* mfcc: Mel Frequency Cepstral Coefficient, represents the short-term power spectrum of a sound
* chroma: Pertains to the 12 different pitch classes
* mel: Mel Spectrogram Frequency

In [6]:
def extract_feature(file_name, mfcc, chroma, mel):
    X, sample_rate = librosa.load(os.path.join(file_name), res_type='kaiser_fast')
    if chroma:
        stft=np.abs(librosa.stft(X))

    result=np.array([])
    if mfcc:
        mfccs=np.mean(librosa.feature.mfcc(y=X, sr=sample_rate, n_mfcc=40).T, axis=0)
        result=np.hstack((result, mfccs))
    if chroma:
        chroma=np.mean(librosa.feature.chroma_stft(S=stft, sr=sample_rate).T,axis=0)
        result=np.hstack((result, chroma))
    if mel:
        mel=np.mean(librosa.feature.melspectrogram(y=X, sr=sample_rate).T,axis=0)
        result=np.hstack((result, mel))
    return result

Now, let’s define a dictionary to hold numbers and the emotions available in the RAVDESS & TESS dataset, and a list to hold all 8 emotions- neutral,calm,happy,sad,angry,fearful,disgust,surprised.

In [9]:
# Emotions in the CREMA Dataset
# Anger, Disgust, Fear, Happy, Neutral, and Sad
emotions={
  'ANG':'anger',
  'HAP':'happy',
  'SAD':'sad',
  'FEA':'fear',
  'DIS':'disgust',
  'NEU':'neutral'
}
# Emotions to observe
observed_emotions=['neutral','happy','sad','anger','fear', 'disgust']

# Load the data and extract features for each sound file

In [21]:
def load_data(test_size=0.2):
    x,y=[],[]
    global path # Ensure 'path' is accessible
    audio_files = glob.glob(os.path.join(path, 'AudioWAV', '*.wav'))
    print(f"Found {len(audio_files)} audio files in {os.path.join(path, 'AudioWAV')}")

    for file in audio_files:
        file_name=os.path.basename(file)
        try:
            emotion_code=file_name.split("_")[2]
            emotion=emotions[emotion_code]
        except (KeyError, IndexError) as e:
            print(f"Skipping file {file_name} due to error in emotion parsing: {e}")
            continue

        if emotion not in observed_emotions:
            print(f"Skipping file {file_name} because emotion '{emotion}' is not in observed_emotions.")
            continue

        feature = get_features_from_file_for_mlp(file)
        if feature is not None: # Ensure feature extraction was successful
            x.append(feature)
            y.append(emotion)
        else:
            print(f"Skipping file {file} due to feature extraction failure (get_features_from_file_for_mlp returned None).")

    if not x:
        print("Warning: No features were extracted. Check dataset path and emotion filtering.")
        # Return empty arrays to prevent ValueError from train_test_split
        return np.array([]), np.array([]), np.array([]), np.array([])

    print(f"Successfully extracted features for {len(x)} files.")
    return train_test_split(np.array(x), y, test_size=test_size, train_size=1-test_size, random_state=9)

# Split the Dataset
Time to split the dataset into training and testing sets! Let’s keep the test set 25% of everything and use the load_data function for this.

In [22]:
# Split the dataset
import time
x_train,x_test,y_train,y_test=load_data(test_size=0.25)

Found 7442 audio files in /root/.cache/kagglehub/datasets/ejlok1/cremad/versions/1/AudioWAV
Successfully extracted features for 7442 files.


#Observe the shape of the training and testing datasets:

In [23]:
#Get the shape of the training and testing datasets
print((x_train.shape[0], x_test.shape[0]))

(5581, 1861)


# Number of features extracted.

In [24]:
# Get the number of features extracted
print(f'Features extracted: {x_train.shape[1]}')

Features extracted: 180


# MLP Classifier

In [25]:
# Initialize the Multi Layer Perceptron Classifier
model=MLPClassifier(alpha=0.01, batch_size=256, epsilon=1e-08, hidden_layer_sizes=(300,), learning_rate='adaptive', max_iter=500)

#Fit/train the model.

In [26]:
# Train the model
model.fit(x_train,y_train)

MLPClassifier(alpha=0.01, batch_size=256, hidden_layer_sizes=(300,),
              learning_rate='adaptive', max_iter=500)

# Predict the accuracy of our model

Let’s predict the values for the test set. This gives us y_pred (the predicted emotions for the features in the test set).

In [27]:
# Predict for the test set
y_pred=model.predict(x_test)

To calculate the accuracy of our model, we’ll call up the accuracy_score() function we imported from sklearn. Finally, we’ll round the accuracy to 2 decimal places and print it out.

In [28]:
# Calculate the accuracy of our model
accuracy=accuracy_score(y_true=y_test, y_pred=y_pred)
# Print the accuracy
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 22.25%


#classification Report

In [29]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))


              precision    recall  f1-score   support

       anger       0.20      0.53      0.29       335
     disgust       0.40      0.10      0.16       304
        fear       0.26      0.13      0.17       327
       happy       0.23      0.52      0.32       299
     neutral       0.00      0.00      0.00       278
         sad       0.17      0.04      0.06       318

    accuracy                           0.22      1861
   macro avg       0.21      0.22      0.17      1861
weighted avg       0.21      0.22      0.17      1861



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Confusion Matrix

In [18]:
from sklearn.metrics import confusion_matrix
matrix = confusion_matrix(y_test,y_pred)
print (matrix)

[[208  18  34  35  29   4]
 [ 22  73  40  25  84  77]
 [ 21  18 147  32  56  49]
 [ 46  38  59  85  79  11]
 [  0  14  23   7 196  36]
 [  0  11  55   4  71 154]]


#Thank You

## Alternative Models for Speech Emotion Recognition

While an `MLPClassifier` can be used for speech emotion recognition, its performance can often be improved by using models better suited for sequential data or more complex patterns. Here are some common alternatives:

1.  **Support Vector Machine (SVM):** A powerful and versatile classification algorithm, often effective with high-dimensional data. SVMs can use different kernel functions to handle non-linear decision boundaries.

2.  **Random Forest:** An ensemble learning method that builds multiple decision trees and merges their predictions to get a more accurate and stable prediction. It's robust to overfitting and can handle a large number of features.

3.  **Convolutional Neural Networks (CNNs):** Excellent for processing grid-like data such as spectrograms (which are essentially 2D images). CNNs can automatically learn hierarchical features from the audio representations.

4.  **Recurrent Neural Networks (RNNs) / Long Short-Term Memory (LSTMs) / Gated Recurrent Units (GRUs):** These are particularly well-suited for sequential data like audio. They can capture temporal dependencies and context within the audio features, which is crucial for emotion recognition.

5.  **Hybrid CNN-RNN Models:** Combining CNNs to extract local spatial features (like patterns in spectrograms) with RNNs/LSTMs to model temporal dependencies can often yield state-of-the-art results in speech emotion recognition.

6.  **XGBoost / LightGBM:** Gradient Boosting models that are highly efficient and often provide excellent performance in various machine learning tasks.

7.  **Transformer Networks:** More recently, Transformer architectures (originally for NLP) have shown great promise in various sequence-to-sequence tasks, including audio processing, by leveraging attention mechanisms to capture long-range dependencies.

In [4]:
!pip install skl2onnx onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 80.1 MB/s eta 0:00:00


In [25]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

initial_type = [('float_input', FloatTensorType([None, x_train.shape[1]]))]

# 3. Convert to ONNX
onx = convert_sklearn(model, initial_types=initial_type, target_opset=15, options={'zipmap': False})

# 4. Save the model
with open("mlp_model.onnx", "wb") as f:
    f.write(onx.SerializeToString())
print("Model saved as mlp_model.onnx")

Model saved as mlp_model.onnx


In [30]:
import onnx

# Load your current model
input_model_path = "mlp_model.onnx"
output_model_path = "mlp_model_sentis_ready.onnx"

# Slice the model from the original input up to the numeric probabilities tensor
onnx.utils.extract_model(
    input_model_path,
    output_model_path,
    input_names=["float_input"],
    output_names=["probabilities"] # Cuts off ArgMax and ArrayFeatureExtractor
)

print("Cleaned model saved successfully!")

Cleaned model saved successfully!


In [26]:
!pip install onnxruntime

In [27]:
import onnxruntime as rt
import numpy as np

# Run inference with onnxruntime
sess = rt.InferenceSession("mlp_model.onnx")
input_name = sess.get_inputs()[0].name
# Convert x_test to float32 as expected by the ONNX model
pred_onx = sess.run(None, {input_name: x_test[:5].astype(np.float32)})[0]
print("Predictions:", pred_onx)

Predictions: ['neutral' 'anger' 'fear' 'happy' 'fear']


In [28]:
print(y_test[:5])

['sad', 'anger', 'fear', 'happy', 'sad']


------------------------------

In [2]:
# Re-running to ensure all dependencies are installed, including onnxscript.
!pip install torch nnAudio onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 606.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.8 MB/s eta 0:00:00


In [3]:
import torch
import onnxscript
import torch.nn as nn
from nnAudio.features.mel import MelSpectrogram
import librosa # Added for file loading in the helper function
import numpy as np # Ensure numpy is available for array conversion

class SentisAudioExtractor(nn.Module):
    def __init__(self):
        super(SentisAudioExtractor, self).__init__()
        # Configure your DSP layer identically to your training parameters
        self.mel_layer = MelSpectrogram(
            sr=16000,       # Match your audio sampling rate
            n_fft=2048,     # FFT window size
            n_mels=60,      # Number of Mel bands
            hop_length=512, # Step size between frames
            trainable_STFT=False # Keep weights frozen (pure DSP processing)
        )

    def forward(self, x):
        # x is raw waveform audio data. Expected shape: [Batch, Samples]
        # e.g., [1, 80000]
        mel_spec = self.mel_layer(x)

        # Convert to Log-scale safely (replicates librosa.power_to_db)
        log_mel_spec = torch.log(torch.clamp(mel_spec, min=1e-10))

        # Flatten the spectrogram to a 1D feature vector per batch item
        flattened = torch.flatten(log_mel_spec, start_dim=1)

        # Slice to ensure it is exactly 180 elements wide
        # This assumes the original MLP was trained on 180 features.
        return flattened[:, :180]

# Instantiate the model globally for use in feature extraction helper
extractor = SentisAudioExtractor()
extractor.eval() # Set to evaluation mode

def get_features_from_file_for_mlp(file_path, target_sr=16000, expected_output_len=180):
    """
    Extracts features from an audio file using the SentisAudioExtractor.

    Args:
        file_path (str): Path to the audio file.
        target_sr (int): The sample rate to resample the audio to (must match extractor).
        expected_output_len (int): The expected length of the feature vector (e.g., 180).

    Returns:
        np.ndarray: A 1D numpy array of extracted features, or None if an error occurs.
    """
    try:
        # Load audio and resample to the target sample rate
        audio, sr = librosa.load(file_path, sr=target_sr, res_type='kaiser_fast')

        # Convert numpy array to torch tensor and add batch dimension
        audio_tensor = torch.from_numpy(audio).unsqueeze(0).float()

        # Ensure the audio length matches expected input length for fixed-size ONNX export
        # Assuming the extractor was designed for 80000 samples based on unity's audioclip sized 80000.
        required_len = 80000
        if audio_tensor.shape[1] < required_len:
            # Pad with zeros
            padding = required_len - audio_tensor.shape[1]
            audio_tensor = torch.nn.functional.pad(audio_tensor, (0, padding))
        elif audio_tensor.shape[1] > required_len:
            # Truncate
            audio_tensor = audio_tensor[:, :required_len]

        # Extract features using the SentisAudioExtractor instance
        with torch.no_grad(): # No need to calculate gradients for feature extraction
            features_tensor = extractor(audio_tensor)

        # Convert the output tensor to a numpy array and remove batch dimension
        features_numpy = features_tensor.squeeze(0).cpu().numpy()

        # Ensure the output length is as expected (e.g., 180 features)
        if features_numpy.shape[0] != expected_output_len:
            print(f"Warning: Extracted features length {features_numpy.shape[0]} does not match expected length {expected_output_len} for file {file_path}. Adjusting.")
            if features_numpy.shape[0] < expected_output_len:
                padded_features = np.zeros(expected_output_len, dtype=features_numpy.dtype)
                padded_features[:features_numpy.shape[0]] = features_numpy
                features_numpy = padded_features
            elif features_numpy.shape[0] > expected_output_len:
                features_numpy = features_numpy[:expected_output_len]

        return features_numpy

    except Exception as e:
        print(f"Error processing file {file_path}: {e}")
        return None

# Keep the ONNX export part as it was requested initially for this cell
dummy_input = torch.randn(1, 80000)

torch.onnx.export(
    extractor,
    dummy_input,
    "AudioFeatureExtractor_Fixed.onnx",
    export_params=True,
    opset_version=18, # Changed opset_version from 15 to 18
    input_names=['raw_audio'],
    output_names=['audio_features'],
    dynamic_axes=None
)

print("Sentis-compatible audio extractor saved successfully!")

STFT kernels created, time used = 0.3754 seconds
STFT filter created, time used = 0.0020 seconds
Mel filter created, time used = 0.0020 seconds
[torch.onnx] Obtain model graph for `SentisAudioExtractor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SentisAudioExtractor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Sentis-compatible audio extractor saved successfully!


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
